# task_05 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

users = pd.read_csv(base / "users.csv")
catalog = pd.read_csv(base / "catalog.csv")
subscriptions = pd.read_csv(base / "subscriptions.csv")
views = pd.read_csv(base / "views.csv")

users["signup_date"] = pd.to_datetime(users["signup_date"])
subscriptions["sub_end"] = pd.to_datetime(subscriptions["sub_end"])
views["view_date"] = pd.to_datetime(views["view_date"])

merged = views.merge(users[["user_id", "plan"]], on="user_id").merge(catalog[["title_id", "genre", "runtime_min", "release_year"]], on="title_id")
merged["completion_rate"] = merged["watch_min"] / merged["runtime_min"]
q1 = str(merged[merged["plan"] == "premium"].groupby("genre")["completion_rate"].mean().sort_values(ascending=False).index[0])

u05 = views[views["user_id"] == "U05"]
daily_counts = u05.groupby("view_date").size().reset_index(name="count")
qualifying = daily_counts[daily_counts["count"] >= 2].sort_values("view_date")
streaks = 0
run_len = 0
prev_date = None
for date in qualifying["view_date"]:
    if prev_date is not None and (date - prev_date).days == 1:
        run_len += 1
    else:
        if run_len >= 3:
            streaks += 1
        run_len = 1
    prev_date = date
if run_len >= 3:
    streaks += 1
q2 = str(int(streaks))

jan_users = set(users[(users["signup_date"].dt.year == 2024) & (users["signup_date"].dt.month == 1)]["user_id"])
q3 = str(int(merged[(merged["user_id"].isin(jan_users)) & (merged["release_year"] < 2020)]["watch_min"].sum()))

ended_users = set(subscriptions[subscriptions["sub_end"] < pd.Timestamp("2024-03-31")]["user_id"])
q4 = str(views[views["user_id"].isin(ended_users)].groupby("title_id")["watch_min"].sum().sort_values(ascending=False).index[0])

doc = views.merge(catalog[["title_id", "genre"]], on="title_id")
total_watch = doc.groupby("view_date")["watch_min"].sum()
doc_watch = doc[doc["genre"] == "Documentary"].groupby("view_date")["watch_min"].sum()
share = (doc_watch / total_watch).fillna(0).reset_index(name="share").sort_values(["share", "view_date"], ascending=[False, True])
q5 = share.iloc[0]["view_date"].date().isoformat()


Q1: Among premium users, which genre has the highest average completion rate, where completion rate = watch_min / runtime_min?

In [ ]:
print(q1)


Q2: For user U05, how many binge streaks are there with at least 3 consecutive days where the user had at least 2 view events on each day?

In [ ]:
print(q2)


Q3: How many total watch minutes came from titles released before 2020 among users whose signup_date is in January 2024?

In [ ]:
print(q3)


Q4: Considering only users whose sub_end is before 2024-03-31, which title_id has the highest total watch_min?

In [ ]:
print(q4)


Q5: On which date was the share of all watch minutes coming from Documentary titles the highest?

In [ ]:
print(q5)
